# Derivatives

Differentiate the parallax light curve with respect to event parameters and
the linear limb coefficient `a1`.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))
import matplotlib.pyplot as plt
from example._common import style
style()


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import singlens as sl


In [ ]:
times = 8000. + jnp.linspace(-40., 40., 1001)
geometry = sl.AnnualParallax(ra_deg=266., dec_deg=-29., t_ref=8000.)
# theta = [t0, u0, t_e, rho, pi_e_n, pi_e_e, a1]
theta = jnp.array([8000., .05, 80., .1, .2, -.1, .5])

def curve(theta):
    return sl.magnification(times, *theta[:3], rho=theta[3], a1=theta[6],
                            parallax=geometry, pi_e_n=theta[4], pi_e_e=theta[5])

A = jax.jit(curve)(theta)
jacobian = jax.jit(jax.jacfwd(curve))(theta)
print(jacobian.shape)  # (observation times, parameters)


In [ ]:
from importlib import import_module
example = import_module("example.jacobian-single.code.autodiff")
fig = example.sensitivity_figure()
plt.show()


Uniform and linear sources share the same geometry. The `a1` derivative
applies to the linear profile. See [numerical accuracy](../numerics.md) when
using derivatives for precision fits.

## Batching and scalar objectives

Use `vmap` for independent events. For a scalar loss, `grad` computes the
parameter gradient without constructing the full light-curve Jacobian.


In [ ]:
events = jnp.stack([theta, theta.at[1].set(.07)])
curves = jax.jit(jax.vmap(curve))(events)
observed = 1.2 * A + .15
error = .01 + .002 * observed

def loss(p):
    return sl.fit_flux(curve(p), observed, error).chi2

gradient = jax.jit(jax.grad(loss))(theta)
print(curves.shape, gradient.shape)


For named derivatives, `jax.jacfwd(model.magnification, argnums=1)` accepts
a `Parameters` object and returns fields such as `.rho`, `.a1`, and `.pi_e_n`.
